# Japanese Audio Subtitles — Colab L4 一键运行面板

无需本地配置 WSL CLI，直接在浏览器选择 **L4 GPU** 运行时即可使用：
1. 挂载 Google Drive（首次运行会自动将模型缓存到 `MyDrive/japanese-subtitles/models/`，后续运行约 1~2 分钟直接恢复）。
2. 指定 Google Drive 中的音频文件夹路径（支持 `.wav` / `.mp3` / `.flac` / `.m4a` 等），生成的中日双语 `.vtt` 字幕将直接输出到原音频同级目录。

In [ ]:
# 1. 检查 L4 GPU 并挂载 Google Drive
import subprocess
from google.colab import drive

print(subprocess.check_output(['nvidia-smi'], text=True))
drive.mount('/content/drive')

In [ ]:
# 2. 拉取仓库代码并恢复/初始化模型（首次运行会下载并缓存到 Google Drive）
import subprocess, sys
from pathlib import Path

if not Path('/content/japanese-audio-subtitles').exists():
    subprocess.run(['git', 'clone', 'https://github.com/cybberJackk/japanese-audio-subtitles.git', '/content/japanese-audio-subtitles'], check=True)
subprocess.run([sys.executable, '-u', '/content/japanese-audio-subtitles/runtime/bootstrap.py'], check=True)

In [ ]:
# 3. 批量转换并生成指定 Google Drive 目录下的全部音频字幕
import json, shutil, subprocess, sys
from pathlib import Path

AUDIO_DIR = Path('/content/drive/MyDrive/your-audio-folder')  # <-- 修改为你的音频文件夹路径
EXTS = {'.wav', '.mp3', '.flac', '.m4a', '.ogg', '.opus', '.aac'}
tracks = [p for p in sorted(AUDIO_DIR.rglob('*')) if p.is_file() and p.suffix.lower() in EXTS and not p.with_name(p.name + '.vtt').exists()]
print(f'待处理音轨数: {len(tracks)}')

manifest = []
work_root = Path('/content/work_jobs')
for idx, src in enumerate(tracks):
    t_dir = work_root / f'{idx:03d}'
    t_dir.mkdir(parents=True, exist_ok=True)
    flac = t_dir / 'input.flac'
    subprocess.run(['ffmpeg', '-nostdin', '-y', '-i', str(src), '-vn', '-ac', '1', '-ar', '16000', '-c:a', 'flac', str(flac)], check=True)
    manifest.append({'index': idx, 'audio': str(flac), 'stem': src.name, 'out_dir': str(t_dir / 'out'), 'metadata_json': ''})

if manifest:
    m_path = work_root / 'batch.json'
    m_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    subprocess.run([sys.executable, '-u', '/content/japanese-audio-subtitles/runtime/pipeline.py', '--batch-json', str(m_path)], check=True)
    for idx, src in enumerate(tracks):
        out_d = work_root / f'{idx:03d}' / 'out'
        for suf in ('.vtt', '.ja.vtt', '.zh.txt', '.ja.txt'):
            f = out_d / (src.name + suf)
            if f.is_file():
                shutil.copy2(f, src.with_name(src.name + suf))
        print('已保存:', src.with_name(src.name + '.vtt'))